<a href="https://colab.research.google.com/github/franciscogarate/mcaf/blob/master/notebooks/Ejercicio_7_PM_Contable_Decesos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/franciscogarate/mcaf

In [ ]:
!pip install pyliferisk

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from pyliferisk import MortalityTable, lx
from pyliferisk.mortalitytables import PASEM2020_Decesos_M_1ord
mt = MortalityTable(qx=PASEM2020_Decesos_M_1ord)

In [ ]:
edad = 50
df = pd.DataFrame(pd.date_range(start='2025-12-31',periods=(mt.w -edad),freq='YE'), columns=['Fecha'])
df['edad'] = edad + df.index
df['t'] = df.index
df['lx'] = df['edad'].apply(lambda x: mt.lx[x+1] if x <= mt.w else 0)
df.head(5)

In [ ]:
df['qx'] = df['lx'].diff(-1).fillna(0)/df['lx'][0]
df.head(5)

In [ ]:
def incr_capital(capital, t):
  return capital * (1 + 0.015) ** t

df['capital'] = incr_capital(5000, df.t)
df.head(5)

In [ ]:
df['pagos'] = df['capital'] * df['qx']
df.head(5)

In [ ]:
df['qx'].sum()

In [ ]:
i = 0.0218
df['factor_desc'] = df['t'].apply(lambda t : 1 / (1 + i) ** t )
df.head(2)

In [ ]:
df.pagos @ df.factor_desc

### Chequeo descuento lineal

In [ ]:
!pip install numpy_financial

In [ ]:
import numpy_financial as npf
npf.npv(i, df.pagos)